In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

In [2]:
#english dataset preprocesing
nltk.download("stopwords")
df = pd.read_csv("engnews.csv")
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str)
stop_words = set(stopwords.words("english"))
def clean_engtext(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)
df["clean_text"] = df["text"].apply(clean_engtext)
df = df[df["clean_text"].str.strip() != ""]
df = df[["clean_text", "FAKE"]]
df.to_csv("clean_english_news_dataset.csv", index=False)
#print("Clean dataset saved")
#print("Total rows:", len(df))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\asmei\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
#Hindi dataset preprocessing
df = pd.read_csv("hindinews.csv", encoding="utf-8")
eng_stop = set(stopwords.words("english"))
hinglish_stop = set(stopwords.words("hinglish"))
stop_words = eng_stop.union(hinglish_stop)
def clean_hinditext(text):
    text = str(text)
    text = transliterate(text, sanscript.DEVANAGARI, sanscript.ITRANS)
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)
df["clean_text"] = df["text"].apply(clean_hinditext)
df = df[["clean_text", "FAKE"]]
df.to_csv("clean_hindi_news_dataset.csv", index=False, encoding="utf-8-sig")
#print("Clean dataset created successfully")

In [4]:
df1 = pd.read_csv("clean_english_news_dataset.csv")
df2 = pd.read_csv("clean_hindi_news_dataset.csv")
merged = pd.concat([df1, df2], ignore_index=True)
merged.to_csv("merged_ds.csv", index=False)
#print("done")

In [5]:
#vectorization of merged dataset
'''df = pd.read_csv("merged_ds.csv")
X = df["clean_text"]
y = df["FAKE"]
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_vectorized = vectorizer.fit_transform(X)
#print("Vectorized shape:", X_vectorized.shape)
#print("vectorization done")'''

'df = pd.read_csv("merged_ds.csv")\nX = df["clean_text"]\ny = df["FAKE"]\nvectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))\nX_vectorized = vectorizer.fit_transform(X)\n#print("Vectorized shape:", X_vectorized.shape)\n#print("vectorization done")'

In [6]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, classification_report

# load dataset
df = pd.read_csv("clean_english_news_dataset.csv")

X = df["clean_text"]
y = df["FAKE"]

# TF-IDF vectorization
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_vec = vectorizer.fit_transform(X)

# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_vec, y, test_size=0.2, random_state=42
)

# SVM model
model = LinearSVC()

model.fit(X_train, y_train)

# predictions
y_pred = model.predict(X_test)

# evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

C:\Users\asmei\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


Accuracy: 0.9960469844138242
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4286
           1       1.00      1.00      1.00      4568

    accuracy                           1.00      8854
   macro avg       1.00      1.00      1.00      8854
weighted avg       1.00      1.00      1.00      8854



In [18]:
reliable_sources = {"bbc.com","reuters.com","nytimes.com","theguardian.com"}
unreliable_sources = {"clickbaitnews.com","fakeportal.com"}
def detect_lang(text):
    # Hindi characters range
    if re.search(r'[\u0900-\u097F]', text):
        return "hindi"
    return "eng"
    
def credibility_score(news_text, source):
    language = detect_lang(news_text)
    if language=="hindi":
        clean= clean_hinditext(news_text)
    elif language=="eng":
        clean = clean_engtext(news_text)
    else:
        print("Error in determining language of the news")

    
    vec = vectorizer.transform([clean])
    fake_prob = model.predict(vec)[0]
    ml_score = 1 - fake_prob

#1.if news from reliable source
    if source in reliable_sources:
        source_score = 1
    elif source in unreliable_sources:
        source_score = 0.1
    else:
        source_score = 0.5
#2.if news already in database
    verified_news = set(df["clean_text"])
    '''if clean in verified_news:
        db_score = 1
    else:
        db_score = 0.5'''

    similar_count = sum(clean in article for article in verified_news)

    if similar_count > 5:
        cross_score = 1
    elif similar_count > 1:
        cross_score = 0.6
    else:
        cross_score = 0.2

    final_score = (0.5 * ml_score +  0.25 * source_score + 0.15 * cross_score)
    #final_score = (0.5 * ml_score +  0.25 * source_score + 0.15 * cross_score + 0.10 * db_score)

    if final_score > 0.7:
        label = "LIKELY TRUE"
    elif final_score > 0.4:
        label = "UNCERTAIN"
    else:
        label = "LIKELY FAKE"

    return {
        "credibility_score": final_score,
        "fake_probability": fake_prob,
        "result": label}

In [19]:
with open("engreal.txt", "r", encoding="utf-8") as f:
        news_text = f.read()
source="bbc.com"
result = credibility_score(news_text, source)
print("engreal:-")
for key,value in result.items():
    print(key," : ",value)

engreal:-
credibility_score  :  0.28
fake_probability  :  1
result  :  LIKELY FAKE


In [20]:
with open("engfake.txt", "r", encoding="utf-8") as f:
        news_text = f.read()
source="bbc.com"
result = credibility_score(news_text, source)
print("engfake:-")
for key,value in result.items():
    print(key," : ",value)

engfake:-
credibility_score  :  0.28
fake_probability  :  1
result  :  LIKELY FAKE


In [21]:
with open("hindireal.txt", "r", encoding="utf-8") as f:
        news_text = f.read()
source="clickbatenews.com"
result = credibility_score(news_text, source)
print("hindireal:-")
for key,value in result.items():
    print(key," : ",value)

hindireal:-
credibility_score  :  0.155
fake_probability  :  1
result  :  LIKELY FAKE


In [22]:
with open("hindifake.txt", "r", encoding="utf-8") as f:
        news_text = f.read()
source="xyz.com"
result = credibility_score(news_text, source)
print("hindifake:-")
for key,value in result.items():
    print(key," : ",value)

hindifake:-
credibility_score  :  0.155
fake_probability  :  1
result  :  LIKELY FAKE
